In [ ]:
# Colab only: select a GPU runtime before running this cell.
import os,sys,subprocess
from pathlib import Path
if not Path('/content').exists():
    raise RuntimeError('Use the main notebook on your Mac; this launcher is for Colab.')
project=Path('/content/movie-sentiment')
if not project.exists():
    subprocess.run(['git','clone','https://github.com/Cameron-Batts/Movie-Review-Sentiment-BERT.git',str(project)],check=True)
os.chdir(project)
# Preserve the runtime's GPU-capable PyTorch build.
subprocess.run([sys.executable,'-m','pip','install','transformers==4.57.1','accelerate==1.11.0','datasets==3.6.0','huggingface-hub==0.36.2'],check=True)


# Movie Review Sentiment Classification with BERT
**Portfolio presentation: Cameron Batts**

Collaborative project. Individual contributions are not specified in the supplied notebook; this presentation does not claim sole authorship.

Python · NLP · Transformers · PyTorch · Naive Bayes · Logistic Regression

## Overview
Compare bag-of-words classifiers with a fully fine-tuned BERT model for binary sentiment classification of Rotten Tomatoes reviews. Preserve the dataset's predefined training, validation and test splits and compare all models on the same 1,066-review test set.

## Business context
Review classification can support content triage and aggregate audience-feedback analysis. This study measures benchmark classification performance rather than a deployed business outcome.

## Data and source identifiers
Hugging Face dataset: `cornell-movie-review-data/rotten_tomatoes`, https://huggingface.co/datasets/cornell-movie-review-data/rotten_tomatoes . Model checkpoint: `MarieAngeA13/Sentiment-Analysis-BERT`. These identifiers are retained for attribution and reproducibility. Raw reviews and person-name examples are not bundled.

## Execution status
Selected numerical outputs are retained from the supplied team notebook. The cleaned code is syntax-checked but not rerun; trained weights are not included. Performance is from one reported run, not independent reproduction.

## Validation update - October 22, 2026
| Model | Accuracy | Macro F1 | Weighted F1 |
| --- | ---: | ---: | ---: |
| Naive Bayes | 0.7777 | 0.7777 | 0.7777 |
| Logistic regression | 0.7636 | 0.7636 | 0.7636 |

The seven traditional-classifier cells reproduced the historical rounded results and the 8,530/1,066/1,066 split. BERT training was not run on this CPU environment. The MPS/CUDA workflow now fixes dataset/model revisions, sets the seed before binary-head initialization, limits checkpoint retention and exports aggregate results. Binary-head and Trainer initialization passed a CPU setup check; accelerator execution and the Colab launcher remain untested. The historical team BERT result of 84.80% remains separate and unverified by a new training run.


In [1]:
# General dataframe imports
import numpy as np
import pandas as pd
pd.set_option("display.max_colwidth", None)

# sklearn imports
from sklearn.metrics import classification_report, accuracy_score
from sklearn import naive_bayes
from sklearn.linear_model import LogisticRegression
from sklearn.feature_extraction.text import CountVectorizer

# Plotting
import matplotlib.pyplot as plt
import seaborn as sns

# Hugging Face/BERT imports
from datasets import load_dataset
import torch
from torch.utils.data import Dataset
from transformers import AutoModelForSequenceClassification, AutoTokenizer, Trainer, TrainingArguments

Executed during validation; source-text and neighbor displays are omitted from saved outputs. See the aggregate validation record.


## Dataset splits
Load the source dataset at runtime. It provides 8,530 training, 1,066 validation and 1,066 test reviews. Training labels are balanced: 4,265 per class; test support is 533 per class. Labels are 0=negative and 1=positive. Dataset and model downloads require internet access.

In [2]:
# Download the dataset
dataset_revision = 'aa13bc287fa6fcab6daf52f0dfb9994269ffea28'
dataset = load_dataset("cornell-movie-review-data/rotten_tomatoes", revision=dataset_revision)

# Store each split as a pandas dataframe
train_df = dataset["train"].to_pandas()
val_df = dataset["validation"].to_pandas()
test_df = dataset["test"].to_pandas()

print("Training observations:", len(train_df))
print("Validation observations:", len(val_df))
print("Testing observations:", len(test_df))

Executed during validation; source-text and neighbor displays are omitted from saved outputs. See the aggregate validation record.


In [3]:
for frame in [train_df, val_df, test_df]:
    frame['sentiment'] = frame['label'].map({0: 'negative', 1: 'positive'})
print(train_df['sentiment'].value_counts())


Executed during validation; source-text and neighbor displays are omitted from saved outputs. See the aggregate validation record.


In [4]:
# Set up X and y for each split
X_train = train_df["text"]
y_train = train_df["label"]

X_val = val_df["text"]
y_val = val_df["label"]

X_test = test_df["text"]
y_test = test_df["label"]

print("X_train:", X_train.shape)
print("X_val:", X_val.shape)
print("X_test:", X_test.shape)

Executed during validation; source-text and neighbor displays are omitted from saved outputs. See the aggregate validation record.


## Bag-of-words representation
Use unigrams and bigrams with English stopword removal. Fit the vocabulary on training text only and reuse it for validation and test transformations. The original vocabulary contains 83,753 features. Stopword removal can suppress sentiment cues such as negation, so its effect deserves validation.

In [5]:
from sklearn.feature_extraction.text import CountVectorizer

vectorizer = CountVectorizer(ngram_range=(1, 2), stop_words="english")

# Fit only on the training data
X_train_vectors = vectorizer.fit_transform(X_train)

# Transform validation and test data using the training vocabulary
X_val_vectors = vectorizer.transform(X_val)
X_test_vectors = vectorizer.transform(X_test)

print("Vocabulary size:", len(vectorizer.get_feature_names_out()))

Executed during validation; source-text and neighbor displays are omitted from saved outputs. See the aggregate validation record.


## Naive Bayes baseline

In [6]:
# Train the Naive Bayes model
model_nb = naive_bayes.MultinomialNB()
model_nb.fit(X_train_vectors, y_train)

# Predict the testing data
y_pred_nb = model_nb.predict(X_test_vectors)

# Display performance
print(classification_report(
    y_test,
    y_pred_nb,
    target_names=["negative", "positive"],
    digits=4,
    zero_division=0.0
))

Executed during validation; source-text and neighbor displays are omitted from saved outputs. See the aggregate validation record.


## Logistic regression baseline

In [7]:
# Train the Logistic Regression model
model_lr = LogisticRegression(max_iter=1000)
model_lr.fit(X_train_vectors, y_train)

# Predict the testing data
y_pred_lr = model_lr.predict(X_test_vectors)

# Display performance
print(classification_report(
    y_test,
    y_pred_lr,
    target_names=["negative", "positive"],
    digits=4,
    zero_division=0.0
))

Executed during validation; source-text and neighbor displays are omitted from saved outputs. See the aggregate validation record.


## BERT setup
Use the checkpoint and tokenizer from the original analysis. CUDA is selected when available, otherwise CPU. The reported run used a Tesla T4 GPU; runtime is hardware-dependent.

In [ ]:
from transformers import set_seed
set_seed(42)  # Set before the new binary classification head is initialized.
bert_path = "MarieAngeA13/Sentiment-Analysis-BERT"
bert_revision = "938fb4267e118e6f892d9825253fd28893d95966"
device = 'mps' if torch.backends.mps.is_available() else ('cuda' if torch.cuda.is_available() else 'cpu')
REQUIRE_ACCELERATOR = globals().get('REQUIRE_ACCELERATOR', True)
if REQUIRE_ACCELERATOR and device == 'cpu':
    raise RuntimeError('Use a Mac with MPS or a GPU runtime for the two-epoch BERT run. Baselines can run on CPU.')
print('Device:', device)
bert_tokenizer = AutoTokenizer.from_pretrained(bert_path, revision=bert_revision)


In [ ]:
# Function to tokenize and convert the data to tensors
def tokenize_and_tensorize(data, labels, tokenizer):
    tokens = tokenizer(
        data.tolist(),
        padding=True,
        truncation=True,
        return_tensors="pt"
    )

    labels_tensor = torch.tensor(labels.tolist())

    return {
        "input_ids": tokens["input_ids"],
        "attention_mask": tokens["attention_mask"]
    }, labels_tensor


train_encodings, y_train_tensor = tokenize_and_tensorize(
    X_train, y_train, bert_tokenizer
)

val_encodings, y_val_tensor = tokenize_and_tensorize(
    X_val, y_val, bert_tokenizer
)

test_encodings, y_test_tensor = tokenize_and_tensorize(
    X_test, y_test, bert_tokenizer
)

In [ ]:
class RottenTomatoesDataset(Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels

    def __getitem__(self, idx):
        item = {
            key: val[idx].clone().detach()
            for key, val in self.encodings.items()
        }

        item["labels"] = self.labels[idx].clone().detach()
        return item

    def __len__(self):
        return len(self.labels)


train_dataset = RottenTomatoesDataset(train_encodings, y_train_tensor)
val_dataset = RottenTomatoesDataset(val_encodings, y_val_tensor)
test_dataset = RottenTomatoesDataset(test_encodings, y_test_tensor)

print("Training:", len(train_dataset))
print("Validation:", len(val_dataset))
print("Testing:", len(test_dataset))

## Binary classification head and training
The source checkpoint has three output classes. Requesting two labels with `ignore_mismatched_sizes=True` reinitializes the incompatible classifier weights and bias while retaining compatible encoder weights. This mismatch is an intentional task adaptation; the new head must be trained.

Fine-tune all parameters for two epochs using learning_rate=2e-5, weight_decay=0.01 and seed=42. Training batch size is 16 and evaluation batch size is 32. Save and evaluate each epoch, then select the checkpoint with lowest validation loss. CUDA runs enable fp16.

In [ ]:
# Load BERT for binary sequence classification
bert_model = AutoModelForSequenceClassification.from_pretrained(
    bert_path,
    revision=bert_revision,
    num_labels=2,
    id2label={0: "negative", 1: "positive"},
    label2id={"negative": 0, "positive": 1},
    ignore_mismatched_sizes=True
)

# Training settings
training_args = TrainingArguments(
    output_dir="./bert_results",
    num_train_epochs=2,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    learning_rate=2e-5,
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    logging_steps=100,
    report_to="none",
    seed=42,
    data_seed=42,
    save_total_limit=2,
    dataloader_pin_memory=(device == "cuda"),
    fp16=torch.cuda.is_available()
)

trainer = Trainer(
    model=bert_model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset
)

In [ ]:
# Fine-tune BERT
trainer.train()

## Held-out evaluation
Evaluate the selected BERT checkpoint on the same predefined test split as the baselines.

In [ ]:
# Predict the test data
bert_output = trainer.predict(test_dataset)

# Convert the model output into predicted labels
y_pred_bert = np.argmax(bert_output.predictions, axis=1)

# Display performance
print(classification_report(
    y_test_tensor.numpy(),
    y_pred_bert,
    target_names=["negative", "positive"],
    digits=4,
    zero_division=0.0
))

              precision    recall  f1-score   support

    negative     0.8857    0.7992    0.8402       533
    positive     0.8171    0.8968    0.8551       533

    accuracy                         0.8480      1066
   macro avg     0.8514    0.8480    0.8477      1066
weighted avg     0.8514    0.8480    0.8477      1066



In [ ]:
# Calculate accuracy for each model
nb_accuracy = accuracy_score(y_test, y_pred_nb)
lr_accuracy = accuracy_score(y_test, y_pred_lr)
bert_accuracy = accuracy_score(y_test_tensor.numpy(), y_pred_bert)

results_df = pd.DataFrame({
    "Model": [
        "Naive Bayes",
        "Logistic Regression",
        "Fine-tuned BERT"
    ],
    "Accuracy": [
        nb_accuracy,
        lr_accuracy,
        bert_accuracy
    ]
})

results_df

                 Model  Accuracy
0          Naive Bayes  0.777674
1  Logistic Regression  0.763602
2      Fine-tuned BERT  0.848030

## Results and interpretation
Fine-tuned BERT reported accuracy 0.848030 (84.80%) and macro F1 0.8477. Naive Bayes reached 0.777674 accuracy and 0.7777 macro F1; logistic regression reached 0.763602 accuracy and 0.7636 macro F1. BERT's accuracy gain was approximately 7.04 percentage points over Naive Bayes and 8.44 points over logistic regression.

BERT's positive-class recall was 0.8968 and negative-class recall was 0.7992, showing an asymmetric error pattern despite balanced test support. The result supports BERT for this experiment, not a universal algorithm ranking.

## Limitations and next steps
- The cleaned workflow was not rerun; dataset/model revisions and package versions are unpinned.
- Only one training seed and dataset split were reported. No confidence interval or significance test was computed.
- Prior checkpoint exposure to related review data has not been audited; independent performance on new reviews is unknown.
- Compare negation-preserving vectorization and tune baseline regularization on validation data.
- Audit errors, evaluate multiple seeds, and compare compute/memory requirements before deployment.

## Contribution context
This is a team-developed analysis presented in Cameron Batts's portfolio. The supplied notebook does not identify individual responsibilities. Shared results should be described as collaborative rather than solely authored.

In [ ]:
# Aggregate results only; no source reviews or model weights are uploaded.
import json, platform
from importlib.metadata import version
from pathlib import Path
project_root = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
record = {
    'python': platform.python_version(), 'device': device,
    'versions': {p: version(p) for p in ['numpy','pandas','scikit-learn','datasets','torch','transformers','accelerate']},
    'seed': 42, 'bert_revision': bert_revision, 'dataset_revision': dataset_revision,
    'train_rows': len(X_train), 'validation_rows': len(X_val), 'test_rows': len(X_test),
    'models': {name: classification_report(y_test, pred, output_dict=True, zero_division=0)
               for name,pred in [('Naive Bayes',y_pred_nb),('Logistic regression',y_pred_lr),('Fine-tuned BERT',y_pred_bert)]},
    'training_logs': trainer.state.log_history,
    'best_validation_checkpoint': trainer.state.best_model_checkpoint,
    'limitations': 'Single predefined split and seed; pretrained training overlap not audited. Collaborative source project; individual responsibilities not established.'
}
destination = project_root/'reports/gpu_rerun_metrics.json'
destination.write_text(json.dumps(record, indent=2)+'\n')
print('Saved:', destination)
